# 📊 Thesis Flight Diagnostics & Envelope Audits

This serves as your **dynamic thesis flight dashboard**. To analyze a specific flight, simply specify the folder name in the **Flight Selection** loader cell below. All subsequent diagnostics cells will automatically parse and plot that selected flight's telemetry!

In [ ]:
# ⚙️ CENTRAL FLIGHT LOADER
# Set SELECTED_FLIGHT to None to automatically load the LATEST flight,
# or enter a specific folder name to load a previous test flight.
SELECTED_FLIGHT = "flight_20260524-1143_75°_column_collision_loop_rotating_cage"

## 🔋 1. Dynamic Flight Envelope & Battery Sag Analysis

In [ ]:
from diagnostics import analyze_flight

# Analyze MoCap tracking errors, active flight bounds, and dynamic battery sag profiles under motor loading
analyze_flight(target=SELECTED_FLIGHT)

## 🔬 2. High-Frequency Telemetry & EKF2 Fusion Rejection Dissection

In [ ]:
import os
from diagnostics import analyze_deep_dissection

# Resolve the path for deep dissection from the selected flight
if SELECTED_FLIGHT:
    bag_path = f"/home/dorten/pi_drone_sshfs/dev_logs/flights/{SELECTED_FLIGHT}/{SELECTED_FLIGHT}_0.mcap"
else:
    bag_path = None
    
analyze_deep_dissection(bag_path=bag_path)

## 📐 3. High-Fidelity Velocity Profiles & EKF2 Convergence Timeline

Visualizes the dual-subplot **smoothed flight velocity magnitude** and **instantaneous /poses stream rate (Hz)**, and displays a timeline auditing **EKF2 absolute coordinate convergence**.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = "/home/dorten/pi_drone_sshfs"
sys.path.append(os.path.join(project_root, "dev_logs", "analysis"))

from dev_logs.analysis.database.db_loader import load_mcap, build_dataframes, load_drone_metadata
from dev_logs.analysis.kinematics.kin_calculator import compute_velocity, find_waypoint_events, build_events_log
from dev_logs.analysis.kinematics.kin_plot_kinematics import plot_velocity_profile, plot_imu_dynamics, plot_imu_xyz_components

# Resolve the active flight folder dynamically
if SELECTED_FLIGHT:
    active_flight = SELECTED_FLIGHT
else:
    import glob
    flight_folders = glob.glob("/home/dorten/pi_drone_sshfs/dev_logs/flights/flight_*")
    flight_folders = [d for d in flight_folders if os.path.isdir(d) and "px4_sd_logs" not in d]
    active_flight = os.path.basename(max(flight_folders, key=os.path.getmtime)) if flight_folders else None

flight_path = os.path.join(project_root, "dev_logs", "flights", active_flight)
print(f"🎬 Performing High-Fidelity Kinematic Diagnostics on: {active_flight}\n")

# Load telemetry dataframes
topic_data, bag_start_ns = load_mcap(flight_path)
drone_tracker_name, _ = load_drone_metadata(project_root)
dfs = build_dataframes(topic_data, drone_tracker_name, bag_start_ns)

df_mocap = dfs['mocap']
df_odom = dfs['odom']
df_setpoint = dfs['setpoint']
df_bat = dfs['battery']
df_imu = dfs['imu']
arming_time = dfs['arming_time']
disarming_time = dfs['disarming_time']

# Process kinematics
df_mocap = compute_velocity(df_mocap)

# Detect Takeoff Trigger (using MoCap Z height crossing 0.15m)
takeoff_mask = df_mocap['z'] > 0.15 if not df_mocap.empty else pd.Series(dtype=bool)
takeoff_time = df_mocap.loc[takeoff_mask, 't'].iloc[0] if takeoff_mask.any() else arming_time + 2.0

# Detect waypoint events dynamically loading from mission definition!
wp_events = find_waypoint_events(df_mocap, takeoff_time, label=active_flight)

# Build dynamic events log
events = build_events_log(df_mocap, df_bat, arming_time, takeoff_time, disarming_time, wp_events)

# Plot Velocity Profile & MoCap /poses Instantaneous Publishing Rate
plot_velocity_profile(df_mocap, wp_events, arming_time, takeoff_time, disarming_time, events, label="Selected Experiment")

# Plot Physical IMU Impact Dynamics
plot_imu_dynamics(df_imu, wp_events, arming_time, takeoff_time, disarming_time, events, label="Selected Experiment")
plot_imu_xyz_components(df_imu, wp_events, arming_time, takeoff_time, disarming_time, events, label="Selected Experiment")

# Align EKF2 and MoCap coordinate timeline to audit convergence
merged = pd.merge_asof(df_mocap.sort_values('t'), df_odom.sort_values('t'), on='t', direction='nearest')
merged = pd.merge_asof(merged, df_setpoint.sort_values('t'), on='t', direction='nearest')

merged['dx_enu'] = merged['x_ekf'] - merged['x']
merged['dy_enu'] = merged['y_ekf'] - merged['y']

print("\n" + "="*70)
print("⚖️ EKF2 ESTIMATOR CONVERGENCE TIMELINE AUDIT")
print("="*70)
timeline_times = [0.0, arming_time, arming_time + 2.0, arming_time + 5.0, arming_time + 10.0, arming_time + 20.0, arming_time + 30.0]
for t in timeline_times:
    if t > merged['t'].max():
        continue
    row = merged.loc[(merged['t'] - t).abs().idxmin()]
    print(f"  t={row['t']-arming_time:>5.2f}s since arming | MoCap ENU: ({row['x']:.3f}, {row['y']:.3f}) | EKF2 ENU: ({row['x_ekf']:.3f}, {row['y_ekf']:.3f}) | EKF2 Offset: ({row['dx_enu']:.3f}, {row['dy_enu']:.3f})")
print("="*70 + "\n")